In [1]:
import numpy as np
from openpi_client import websocket_client_policy

client = websocket_client_policy.WebsocketClientPolicy(host="localhost", port=18000)
print("Server metadata:", client.get_server_metadata())

observation = {
    "observation/exterior_image_1_left": np.random.randint(256, size=(224, 224, 3), dtype=np.uint8),
    "observation/wrist_image_left": np.random.randint(256, size=(224, 224, 3), dtype=np.uint8),
    "observation/joint_position": np.random.rand(7),
    "observation/gripper_position": np.random.rand(1),
    "prompt": "pick up the cube and place it in the basket",
}

result = client.infer(observation)
actions = np.asarray(result["actions"])
print("Action chunk shape:", actions.shape)   # expect (15, 8)
print("First action:", actions[0])

Server metadata: {}
Action chunk shape: (15, 8)
First action: [ 0.00692769 -0.01010163 -0.00638996  0.00701539  0.00604718 -0.0126149
  0.00158574  0.03772453]


In [3]:
import os, io, requests, numpy as np
from PIL import Image

CAMERA_URL = os.getenv("CAMERA_SERVICE_URL", "http://127.0.0.1:54322")
FRANKY_URL = os.getenv("FRANKY_SERVICE_URL", "http://127.0.0.1:54321")

# camera_service camera_id == str(int(serial))  (leading zeros stripped)
LEFT_CAM  = str(int(os.environ["LEFT_CAMERA_SERIAL"]))
WRIST_CAM = str(int(os.environ["WRIST_CAMERA_SERIAL"]))
# or discover: requests.get(f"{CAMERA_URL}/cameras").json()

def get_rgb(cam_id, q=90):
    r = requests.get(f"{CAMERA_URL}/camera/{cam_id}/rgb.jpg",
                     params={"jpeg_quality": q}, timeout=5)
    r.raise_for_status()
    return np.asarray(Image.open(io.BytesIO(r.content)).convert("RGB"), dtype=np.uint8)

def get_joints():
    r = requests.get(f"{FRANKY_URL}/joint_state", timeout=5)
    r.raise_for_status()
    return np.asarray(r.json()["positions"], dtype=np.float32)  # (7,)


In [4]:
left  = get_rgb(LEFT_CAM)
wrist = get_rgb(WRIST_CAM)
q     = get_joints()
grip  = 0.0   # gripper_service isn't running; fine for a single test
print(left.shape, wrist.shape, q, grip)


(480, 640, 3) (480, 640, 3) [ 5.7782527e-06  1.4503019e-04  3.7766267e-06 -1.5709087e+00
 -6.6348998e-06  1.5709904e+00 -1.0935401e-04] 0.0


In [16]:
from openpi_client import image_tools, websocket_client_policy

client = websocket_client_policy.WebsocketClientPolicy(host="localhost", port=18000)

observation = {
    "observation/exterior_image_1_left": image_tools.resize_with_pad(left, 224, 224),
    "observation/wrist_image_left":      image_tools.resize_with_pad(wrist, 224, 224),
    "observation/joint_position":        q,
    "observation/gripper_position":      np.array([grip], dtype=np.float32),
    "prompt": "pick up the white color object from the table",
}
actions = np.asarray(client.infer(observation)["actions"])
print(actions.shape, actions[0])   # (15, 8): 7 joint targets + gripper


(15, 8) [ 0.22034504 -0.05830436  0.02242071 -0.11922816  0.01025693  0.05074435
  0.23458527  0.01743154]


In [23]:
import numpy as np, time, requests
FRANKY_URL = "http://127.0.0.1:54321"
Q_MIN = np.array([-2.7437,-1.7837,-2.9007,-3.0421,-2.8065,0.5445,-3.0159])
Q_MAX = np.array([ 2.7437, 1.7837, 2.9007,-0.1518, 2.8065,4.5169, 3.0159])

CONTROL_HZ    = 15.0
EXECUTE_STEPS = 8
VEL_CAPS      = np.full(7, 0.10)      # rad/s, per joint  (msbutt1 default)
CLOSE_TH, RELEASE_TH = 0.50, 0.10
dt = 1.0 / CONTROL_HZ

def get_q():   return np.asarray(requests.get(f"{FRANKY_URL}/joint_state", timeout=5).json()["positions"], float)
def send_q(q, seq): requests.post(f"{FRANKY_URL}/target_joint_state",
                    json={"positions": np.clip(q,Q_MIN,Q_MAX).tolist(),"velocities":[0.0]*7,"seq":int(seq)}, timeout=5).raise_for_status()
def stop():    requests.post(f"{FRANKY_URL}/stop", timeout=5)

# actions = (16,8) from client.infer(...); col 7 = gripper closedness
v = np.clip(np.asarray(actions)[:EXECUTE_STEPS, :7], -VEL_CAPS, VEL_CAPS)
grip_plan = np.asarray(actions)[:EXECUTE_STEPS, 7]

q0 = get_q()
q_traj = q0 + np.cumsum(v * dt, axis=0)                 # integrate velocity -> position targets

print("chunk motion (rad):", np.round(np.abs(q_traj[-1]-q0), 3), " gripper:", np.round(grip_plan,2))
assert (q_traj >= Q_MIN).all() and (q_traj <= Q_MAX).all(), "chunk leaves joint limits"
assert np.abs(q_traj[-1] - q0).max() < 0.3, "chunk motion larger than expected"
if grip_plan.max() >= CLOSE_TH:  input("policy wants to CLOSE — Enter to allow, Ctrl+C to abort")

requests.post(f"{FRANKY_URL}/command_timeout", json={"command_timeout_s": 3.0})
try:
    for k, qk in enumerate(q_traj):
        send_q(qk, k); time.sleep(dt)
    for _ in range(5): send_q(q_traj[-1], 999); time.sleep(dt)   # hold vs watchdog
finally:
    stop(); print("final q:", np.round(get_q(), 3))


chunk motion (rad): [0.053 0.019 0.006 0.043 0.007 0.028 0.053]  gripper: [0.02 0.01 0.01 0.01 0.01 0.02 0.01 0.01]
final q: [ 0.102 -0.036  0.011 -1.653  0.014  1.625  0.102]
